# Image Moderation (TF NSFW)

# Image Moderation — TF NSFW classifier (replaces NudeNet baseline)

Trains on the local NSFW corpus (`data/nsfw/{train,val,test}`) plus the Reddit
title→is_nsfw table as a text-prior. Serves the `app/moderation_engine.py`
`/api/v1/moderation/image` endpoint; keep p(NSFW) calibration + a
contamination-tolerant eval split.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}
from tf_utils import set_memory_growth
set_memory_growth()


In [ ]:
from buddy_data import nsfw_images, reddit_nsfw
root = nsfw_images()
print('train dir:', root / 'train')
nsfw_df = reddit_nsfw()
print('reddit titles:', nsfw_df.shape, nsfw_df['is_nsfw'].value_counts().to_dict())

In [ ]:
import tensorflow as tf
IMG = 224
# ImageFolder-style: read local train/val dirs; each class a subdir.
train_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'train', image_size=(IMG, IMG), batch_size=32, shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'val', image_size=(IMG, IMG), batch_size=32)
train_ds = train_ds.map(lambda x, y: (x / 255.0, y)).prefetch(tf.data.AUTOTUNE)

In [ ]:
import tensorflow as tf
base = tf.keras.applications.MobileNetV3Small(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False
x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
m = tf.keras.Model(base.input, out)
m.compile('adam', 'binary_crossentropy', metrics=['accuracy', 'AUC'])
m.summary()

In [ ]:
m.fit(train_ds, validation_data=val_ds, epochs=5)
# Fine-tune last 40 layers at 1e-5 once the head converges.

In [ ]:
# Report ROC-AUC + precision@k on val_ds. Target: beat NudeNet baseline.

In [ ]:
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log
onnx = export_keras_onnx(m, Path('../models'), 'nsfw_classifier', '1.0.0')
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name':'nsfw_classifier','version':'1.0.0','artifact_path':str(q),
            'framework':'tensorflow','metrics':{'auc':0.0}})